# 05 — Behavioral Calibration (GRPO)

GRPO reward training: reward correct+confident & 'I don't know'+truly-unanswerable, penalize confidently wrong.

**Papers**
- Behavioral Calibration https://arxiv.org/abs/2512.19920
- TRL GRPOTrainer (docs)

In [ ]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])

from utils.colab import bootstrap
bootstrap()


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")


## 1. Setup & config

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, load_model_unsloth, apply_qlora, count_trainable_params,
    cuda_device_info,
)
from experiments.shared.eval_utils import compute_perplexity, benchmark_speed, vram_snapshot
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve


In [ ]:
set_env(quiet=True)
tracking, tags, run_name = setup_run(
    experiment='behavioral_calibration',
    model=DEFAULT_MODEL_NAME,
    stage="experiment",
    mlflow_experiment_name='burnit-bg-experiments',
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")


## 2. Data loading

In [ ]:
# ###### Pick a dataset from MinIO (or fall back to local data_prep/) ######
# Set DATASET_PREFIX in .env / Colab secrets to skip the picker, e.g.
#   DATASET_PREFIX=data_prep/processed/mental-health
# Or pass `auto=True` below for non-interactive runs.

DEFAULT_PREFIX = os.getenv("DATASET_PREFIX", "data_prep/processed")
try:
    local_dataset_dir = resolve(prefix=DEFAULT_PREFIX, auto=False)
    TRAIN_PATH = local_dataset_dir / "train.jsonl"
    EVAL_PATH  = local_dataset_dir / "eval.jsonl"
except (FileNotFoundError, Exception) as exc:
    print(f"[data] MinIO lookup failed ({exc}); falling back to local data_prep/processed/")
    TRAIN_PATH = REPO_ROOT / "data_prep/processed/train.jsonl"
    EVAL_PATH  = REPO_ROOT / "data_prep/processed/eval.jsonl"

train_records = list(load_alpaca_dataset(TRAIN_PATH))
eval_records  = list(load_alpaca_dataset(EVAL_PATH))
train_stats = dataset_statistics(train_records)
eval_stats  = dataset_statistics(eval_records)
print(f"train: {len(train_records)} rows  eval: {len(eval_records)} rows")
print(train_stats)


## 2b. Reward function

In [ ]:
REFUSAL_MARKERS = ("не съм сигурен", "не знам", "i don't know", "i am not sure")

def reward_fn(prompts, completions, golds, **_):
    rewards = []
    for prompt, completion, gold in zip(prompts, completions, golds):
        text = completion.lower().strip()
        refused = any(marker in text for marker in REFUSAL_MARKERS)
        looks_right = any(tok in text for tok in gold.lower().split()[:6])
        if looks_right and not refused: rewards.append(1.0)
        elif refused and not looks_right: rewards.append(0.4)
        elif refused and looks_right: rewards.append(-0.3)
        elif looks_right: rewards.append(0.5)
        else: rewards.append(-1.0)
    return rewards


## 3. GRPO training

In [ ]:
# trl.GRPOTrainer generates K completions per prompt, then applies a
# group-relative advantage update using the reward_fn above.
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

grpo_prompts = [{"prompt": format_prompt(r["instruction"], template="alpaca"),
                 "gold": r["output"]} for r in train_records[:200]]
ds = Dataset.from_list(grpo_prompts)

OUTPUT_DIR = REPO_ROOT / "tmp/experiments/behavioral_calibration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
grpo_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR), num_generations=4, num_train_epochs=1,
    learning_rate=5e-6, logging_steps=5, bf16=True,
    per_device_train_batch_size=1, gradient_accumulation_steps=4,
    report_to=[],
)
with tracking.run(run_name=run_name, tags=tags):
    with stage(tracking, "grpo_train"):
        trainer = GRPOTrainer(
            model=model, tokenizer=tokenizer,
            reward_funcs=lambda prompts, completions, **kw: reward_fn(
                prompts, completions, kw.get("gold", [""] * len(prompts))
            ),
            args=grpo_args, train_dataset=ds,
        )
        trainer.train()


## 4. Reward distribution before vs after

In [ ]:
import numpy as np
@torch.no_grad()
def sample_rewards(records, n=32):
    rs = []
    for r in records[:n]:
        out = generate(r["instruction"]) if 'generate' in dir() else ""
        rs.extend(reward_fn([r["instruction"]], [out], [r["output"]]))
    return np.array(rs)

rewards_after = sample_rewards(eval_records, n=32)
tracking.log_metrics({
    "reward.mean": float(rewards_after.mean()),
    "reward.std":  float(rewards_after.std()),
})
print(f"reward mean={rewards_after.mean():.3f} std={rewards_after.std():.3f}")


## 6. Inference test

In [ ]:
# ###### Inference test (mental-health prompts) ######
with stage(tracking, "inference_test"):
    batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
    log_responses(
        tracking,
        experiment='behavioral_calibration',
        model_checkpoint=str(REPO_ROOT / "tmp/experiments/behavioral_calibration"),
        **batteries,
    )
    for k, v in batteries.items():
        print(f"-- {k} --")
        for entry in v[:2]:
            print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")


## 7. Save via MLflow + GGUF export

In [ ]:
# ###### Save model via MLflow (single source of truth) ######
# tracking.log_model logs the model artifact under runs:/<id>/model and,
# when registered_model_name is set, adds a new version to the Models tab.
# MLflow's artifact store backs onto MinIO — no separate upload needed.
with stage(tracking, "save_model"):
    try:
        tracking.log_model(
            model,
            flavor="transformers",
            artifact_path="model",
            registered_model_name='burnit-bg-behavioral-calibration',
            input_example=None,
        )
        print("[save] model logged via MLflow")
    except Exception as exc:
        print(f"[save] log_model failed: {exc}")

# Optional: GGUF export for offline local inference (RTX 3050 / Ollama).
# The GGUF is added as a run artifact under `gguf/`.
with stage(tracking, "gguf_export"):
    try:
        from experiments.shared.model_utils import export_gguf
        gguf_path = export_gguf(model, tokenizer, REPO_ROOT / "tmp/experiments/behavioral_calibration/gguf", quantization="q4_k_m")
        tracking.save_data(gguf_path, artifact_path="gguf")
        print(f"[save] GGUF logged: {gguf_path}")
    except Exception as exc:
        print(f"[save] GGUF export skipped: {exc}")
